# Chatbot GPT from scratch

## Model

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, IterableDataset
import torch.nn.init as init
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
torch.device("cuda" if torch.cuda.is_available() else "cpu")

device(type='cuda')

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros((max_seq_length, d_model))
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "Vetor de embedding precisa ser divisivel pelo número de cabeças da camada de atenção!"
        self.head_dim = d_model // num_heads
        self.d_model, self.num_heads = d_model, num_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.output_linear = nn.Linear(d_model, d_model)

    def split_heads(self, x, encoder_output=None):
        # Entra Q, K, V com dimensão (batch_size, sequence_length, d_model)
        # Reshape para (batch_size, sequence_length, num_heads, d_model)
        # Reordering para (batch_size, num_heads, sequence_length, d_model)
        if encoder_output is None:
            x = torch.reshape(x, shape=(x.shape[0], x.shape[1], self.num_heads, self.head_dim)) #.contiguous()
            x = x.permute(0, 2, 1, 3)
        else:
            raise NotImplementedError("Modelo ainda não compatível com Encoder.")
        return x

    def compute_attention_scores(self, q_linear_out, k_linear_out, v_linear_out, mask=None):
        qk_dot_product = torch.matmul(q_linear_out, k_linear_out.transpose(2, 3)) / self.head_dim ** 0.5

        if mask is not None:
            qk_dot_product = qk_dot_product.masked_fill(mask == 0, float('-inf'))

        attn_scores = nn.functional.softmax(qk_dot_product, dim=-1)
        attn_weighted_v = torch.matmul(attn_scores, v_linear_out)

        return attn_weighted_v


    def combine_heads(self, x):
        x = x.permute(0, 2, 1, 3).contiguous()
        return torch.reshape(x, shape=(x.shape[0], x.shape[1], int(x.shape[2] * x.shape[3])))

    def forward(self, x, mask):
        q_linear_out = self.split_heads(self.q(x))
        k_linear_out = self.split_heads(self.k(x))
        v_linear_out = self.split_heads(self.v(x))
        
        attn_weighted_v = self.compute_attention_scores(q_linear_out, k_linear_out, v_linear_out, mask=mask)
        attn_weighted_v = self.combine_heads(attn_weighted_v)
        return self.output_linear(attn_weighted_v)


class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, hidden_size):
        super().__init__()
        self.ff_1 = nn.Linear(d_model, hidden_size)
        self.ff_2 = nn.Linear(hidden_size, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.ff_2(self.relu(self.ff_1(x)))
    

class DecoderBlock(nn.Module):
    def __init__(self, d_model, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.feed_forward = FeedForwardSubLayer(d_model, hidden_size)
        self.mha = MultiHeadAttention(d_model, num_heads) # nn.MultiheadAttention()
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, tgt_mask):
        x = self.norm_1(x + self.dropout(self.mha(x, mask=tgt_mask)))
        x = self.norm_2(x + self.dropout(self.feed_forward(x)))
        return x
    

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, max_sequence_length, n_layers, hidden_size, num_heads, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=0)
        self.pe = PositionalEncoding(d_model, max_sequence_length)
        self.layers = nn.ModuleList(
            [DecoderBlock(d_model, hidden_size, num_heads, dropout) for _ in range(n_layers)]
        )
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x, tgt_mask):
        x = self.embedding(x)
        x = self.pe(x)
        for layer in self.layers:
            x = layer(x, tgt_mask)
        out = self.output_layer(x)
        return out


## Dataset Prep

### Tokenizer Char Level

In [4]:
class TokenizerChar:
    def __init__(self):
        self.chr_to_idx = {chr(v): v for v in range(1, 257)}
        self.chr_to_idx['<SOS>'] = 257
        self.chr_to_idx['<EOS>'] = 258
        self.chr_to_idx['<PAD>'] = 0
        self.chr_to_idx['<UNK>'] = 259
        self.chr_to_idx['<EOP>'] = 260
        self.special_tokens_chrs = ['<SOS>', '<EOS>', '<PAD>', '<UNK>', '<EOP>']

        self.idx_to_chr = {v: k for k, v in self.chr_to_idx.items()}

        self.vocab_size = len(self.chr_to_idx.keys())

    def encode(self, char):
        if char in self.chr_to_idx.keys():
            return self.chr_to_idx[char]
        else:
            return 259
        
    def encode_text(self, text):
        tokens = []
        i = 0
        while i < len(text):
            # Detecta tokens especiais (todos têm 5 caracteres e começam com '<')
            if len(text) - i >= 5:
                if text[i] == '<' and i + 4 < len(text) and text[i:i+5] in self.special_tokens_chrs:
                    tokens.append(self.encode(text[i:i+5]))
                    i += 5
                else:
                    tokens.append(self.encode(text[i]))
                    i += 1
            else:
                tokens.append(self.encode(text[i]))
                i += 1
        return tokens
    
    def decode(self, token_idx):
        return self.idx_to_chr[token_idx]
    
    def sos_token(self):
        return '<SOS>'
    
    def sos_token_idx(self):
        return self.chr_to_idx['<SOS>']

    def eos_token(self):
        return '<EOS>'
    
    def eos_token_idx(self):
        return self.chr_to_idx['<EOS>']
    
    def pad_token(self):
        return '<PAD>'
    
    def pad_token_idx(self):
        return self.chr_to_idx['<PAD>']
    
    def eop_token(self):
        return '<EOP>'
    
    def eop_token_idx(self):
        return self.chr_to_idx['<EOP>']
    
    def get_vocab_size(self):
        return self.vocab_size

### Tokenizer Byte Pair

In [ ]:
from collections import defaultdict
from typing import List


class TrieText:
    def __init__(self):
        self.trie = defaultdict(dict)
    
    def get_trie(self):
        return self.trie
    
    def add_node(self, parent_nodes: List[str], new_node: str, valid_token: bool = True):
        trie = self.trie

        if parent_nodes == []:
            trie[new_node] = defaultdict(dict)
            return
        
        last_node = ''
        for node in parent_nodes:
            trie = trie[node]
            last_node = node

        if last_node in new_node:
            trie[new_node] = defaultdict(dict)
            trie['_end_'] = valid_token
        else:
            raise ValueError("New node is not valid! It must contain previous parent node string values.")
    
    def get_node_path(self, node: str, get_parent_nodes=False):
        nodes = []
        current_node = self.trie.copy()
        for i in range(1, len(node) + 1):
            key = node[:i]
            if key in current_node.keys():
                nodes.append(key)
                current_node = current_node[key]
            else:
                break
        if get_parent_nodes:
            return nodes[:-1]
        return nodes

In [65]:
trie = TrieText()

In [66]:
trie.add_node([], "a")

In [69]:
trie.get_node_path("a", get_parent_nodes=False)

['a']

In [72]:
trie.add_node(["a"], "ac")

In [74]:
trie.get_node_path("ac")

['a', 'ac']

In [75]:
trie.add_node(["a"], "bc")

ValueError: New node is not valid! It must contain previous parent node string values.

In [ ]:
class TokenizerBPE:
    def __init__(self, vocab_size=50_000):
        self.token_to_idx = dict()
        self.token_to_idx['<PAD>'] = 0
        self.token_to_idx['<SOS>'] = 1
        self.token_to_idx['<EOS>'] = 2
        self.token_to_idx['<UNK>'] = 3
        self.token_to_idx['<EOP>'] = 4
        self.special_tokens = ['<SOS>', '<EOS>', '<PAD>', '<UNK>', '<EOP>']
        self.idx_to_token = {v: k for k, v in self.token_to_idx.items()}
        self.vocab_size = vocab_size

    def fit(self, text_corpus):
        return

    def encode(self, token):
        return
        
    def encode_text(self, text):
        return
    
    def decode(self, token_idx):
        return self.idx_to_token[token_idx]
    
    def sos_token(self):
        return '<SOS>'
    
    def sos_token_idx(self):
        return self.token_to_idx['<SOS>']

    def eos_token(self):
        return '<EOS>'
    
    def eos_token_idx(self):
        return self.token_to_idx['<EOS>']
    
    def pad_token(self):
        return '<PAD>'
    
    def pad_token_idx(self):
        return self.token_to_idx['<PAD>']
    
    def eop_token(self):
        return '<EOP>'
    
    def eop_token_idx(self):
        return self.token_to_idx['<EOP>']
    
    def get_vocab_size(self):
        return self.vocab_size

In [ ]:
text_sample = "O rato roeu a roupa do rei de Roma. A rainha de Portugal gosta de queijo e pão. <SOS> Era uma vez um gato preto. <EOS>"
tokenizer_bpe = TokenizerBPE(vocab_size=50)
tokenizer_bpe.fit(text_sample)

pairs=['O ', ' r', 'ra', 'at', 'to', 'o ', ' r', 'ro', 'oe', 'eu', 'u ', ' a', 'a ', ' r', 'ro', 'ou', 'up', 'pa', 'a ', ' d', 'do', 'o ', ' r', 're', 'ei', 'i ', ' d', 'de', 'e ', ' R', 'Ro', 'om', 'ma', 'a.', '. ', ' A', 'A ', ' r', 'ra', 'ai', 'in', 'nh', 'ha', 'a ', ' d', 'de', 'e ', ' P', 'Po', 'or', 'rt', 'tu', 'ug', 'ga', 'al', 'l ', ' g', 'go', 'os', 'st', 'ta', 'a ', ' d', 'de', 'e ', ' q', 'qu', 'ue', 'ei', 'ij', 'jo', 'o ', ' e', 'e ', ' p', 'pã', 'ão', 'o.', '. ', ' <', '<S', 'SO', 'OS', 'S>', '> ', ' E', 'Er', 'ra', 'a ', ' u', 'um', 'ma', 'a ', ' v', 've', 'ez', 'z ', ' u', 'um', 'm ', ' g', 'ga', 'at', 'to', 'o ', ' p', 'pr', 're', 'et', 'to', 'o.', '. ', ' <', '<E', 'EO', 'OS', 'S>']
most_frequent_bp=[('a ', 6)]
pairs=['O ', ' r', 'ra', 'at', 'to', 'o ', ' r', 'ro', 'oe', 'eu', 'u ', ' a', 'a ', ' r', 'ro', 'ou', 'up', 'pa', 'a ', ' d', 'do', 'o ', ' r', 're', 'ei', 'i ', ' d', 'de', 'e ', ' R', 'Ro', 'om', 'ma', 'a.', '. ', ' A', 'A ', ' r', 'ra', 'ai', 'in', 'nh', 'ha

### Dataset

#### Pre-training dataset

In [ ]:
import json
from torch.utils.data import Dataset

In [ ]:
dataset = pd.read_parquet('dataset_text/wiki_pt_0-50k.parquet')

In [ ]:
dataset.head()

,id,url,title,text
0,220,https://pt.wikipedia.org/wiki/Astronomia,Astronomia,Astronomia é uma ciência natural que estuda co...
1,223,https://pt.wikipedia.org/wiki/Am%C3%A9rica%20L...,América Latina,A América Latina (; ) é uma região do continen...
2,224,https://pt.wikipedia.org/wiki/Albino%20Forjaz%...,Albino Forjaz de Sampaio,Albino Maria Pereira Forjaz de Sampaio (Lisboa...
3,226,https://pt.wikipedia.org/wiki/Anno%20Domini,Anno Domini,Anno Domini (A.D.) é uma expressão em latim qu...
4,228,https://pt.wikipedia.org/wiki/Aquiles,Aquiles,"Aquiles (), na mitologia grega, foi um herói d..."


In [ ]:
np.random.seed(42)  # Para reprodutibilidade
dataset['split'] = np.random.choice(['train', 'test'], size=len(dataset), p=[0.99, 0.01])

In [ ]:
dataset.head()

,id,url,title,text,split
0,220,https://pt.wikipedia.org/wiki/Astronomia,Astronomia,Astronomia é uma ciência natural que estuda co...,train
1,223,https://pt.wikipedia.org/wiki/Am%C3%A9rica%20L...,América Latina,A América Latina (; ) é uma região do continen...,train
2,224,https://pt.wikipedia.org/wiki/Albino%20Forjaz%...,Albino Forjaz de Sampaio,Albino Maria Pereira Forjaz de Sampaio (Lisboa...,train
3,226,https://pt.wikipedia.org/wiki/Anno%20Domini,Anno Domini,Anno Domini (A.D.) é uma expressão em latim qu...,train
4,228,https://pt.wikipedia.org/wiki/Aquiles,Aquiles,"Aquiles (), na mitologia grega, foi um herói d...",train


In [ ]:
dataset.text[1].split('\n\n')[0][:2000]

'A América Latina (; ) é uma região do continente americano que engloba os países onde são faladas, primordialmente, línguas românicas (derivadas do latim) — no caso, o espanhol, o português e o francês — visto que, historicamente, a região foi maioritariamente dominada pelos impérios coloniais europeus Espanhol e Português. A América Latina tem uma área de cerca de  km², o equivalente a cerca de 3,9% da superfície da Terra (ou 14,1% de sua superfície emersa terrestre). Em 2008, a sua população foi estimada em mais de 569 milhões de pessoas. Os países do restante do continente americano tiveram uma colonização majoritariamente realizada por povos europeus de cultura anglo-saxônica ou neerlandesa (ver América Anglo-Saxônica). Vale ressaltar algumas exceções, como Québec, que não é um país independente, mas uma província de maioria francófona que pertence ao Canadá; o estado da Luisiana, que também foi colonizado por franceses, mas pertence aos Estados Unidos e os estados do sudoeste est

In [ ]:
def df_wiki_prep(df_wiki, sequence_length=512, overlap=256, tokenizer=TokenizerChar()):
    text_items, ids, urls, titles, splits = [], [], [], [], []
    sos = tokenizer.sos_token()  # '<SOS>'
    eos = tokenizer.eos_token()  # '<EOS>'
    for instance_idx in tqdm(range(df_wiki.shape[0]), desc='Generating training instances...'):
        title = df_wiki.title[instance_idx]
        text = sos + df_wiki.text[instance_idx] + eos
        id = df_wiki.id[instance_idx]
        url = df_wiki.url[instance_idx]
        split = df_wiki.split[instance_idx]
        text_length = len(text)
        start = 0
        while start + sequence_length < text_length:
            sliced_text = f'Tema: {title}. Texto: ' + text[start:start + sequence_length]
            text_items.append(sliced_text)
            ids.append(id)
            urls.append(url)
            titles.append(title)
            splits.append(split)
            start += sequence_length - overlap
        if text_length - start >= 256:
            sliced_text = text[start:text_length]
        elif text_length > sequence_length:
            sliced_text = text[-256:]
        else:
            sliced_text = text
        text_items.append(sliced_text)
        ids.append(id)
        urls.append(url)
        titles.append(title)
        splits.append(split)
    df_wiki_preped = pd.DataFrame(
        {
            'id': ids,
            'urls': urls,
            'titles': titles,
            'split': splits,
            'text': text_items
        }
    )
    return df_wiki_preped

In [ ]:
df_wiki_preped = df_wiki_prep(dataset, sequence_length=1024, overlap=256)

Generating training instances...: 100%|██████████| 50000/50000 [00:01<00:00, 26020.65it/s]


In [ ]:
df_wiki_preped.text[0]

'Tema: Astronomia. Texto: <SOS>Astronomia é uma ciência natural que estuda corpos celestes (como estrelas, planetas, cometas, nebulosas, aglomerados de estrelas, galáxias) e fenômenos que se originam fora da atmosfera da Terra (como a radiação cósmica de fundo em micro-ondas). Preocupada com a evolução, a física e a química de objetos celestes, bem como a formação e o desenvolvimento do universo.\n\nA astronomia é uma das mais antigas ciências. Culturas pré-históricas deixaram registrados vários artefatos astronômicos, como Stonehenge, os montes de Newgrange e os menires. As primeiras civilizações, como os babilônios, gregos, chineses, indianos, persas e maias realizaram observações metódicas do céu noturno. No entanto, a invenção do telescópio permitiu o desenvolvimento da astronomia moderna. Historicamente, a astronomia incluiu disciplinas tão diversas como astrometria, navegação astronômica, astronomia observacional e a elaboração de calendários. Durante o período medieval, seu estu

In [ ]:
df_wiki_preped.text[1]

'Tema: Astronomia. Texto: moderna. Historicamente, a astronomia incluiu disciplinas tão diversas como astrometria, navegação astronômica, astronomia observacional e a elaboração de calendários. Durante o período medieval, seu estudo era obrigatório e estava incluído no Quadrivium que, junto com o Trivium, compunha a metodologia de ensino das sete Artes liberais.\n\nDurante o século XX, o campo da astronomia profissional dividiu-se em dois ramos: a astronomia observacional e a astronomia teórica. A primeira está focada na aquisição de dados a partir da observação de objetos celestes, que são então analisados utilizando os princípios básicos da física. Já a segunda é orientada para o desenvolvimento de modelos analíticos que descrevem objetos e fenômenos astronômicos. Os dois campos se complementam, com a astronomia teórica procurando explicar os resultados observacionais, bem com as observações sendo usadas para confirmar (ou não) os resultados teóricos.\n\nOs astrônomos amadores têm co

In [ ]:
df_wiki_preped[df_wiki_preped['split'] == 'train'].to_parquet('dataset_text/wiki_pt_0-50k_train.parquet')
df_wiki_preped[df_wiki_preped['split'] == 'test'].to_parquet('dataset_text/wiki_pt_0-50k_test.parquet')

In [ ]:
tokenizer = TokenizerChar()

In [ ]:
tokenizer.encode_text('<SOS>')

[257]

In [ ]:
tokenizer.encode_text('<SOS> ')

[257, 32]

In [ ]:
print([tokenizer.decode(t) for t in tokenizer.encode_text(df_wiki_preped.text[0])])

['T', 'e', 'm', 'a', ':', ' ', 'A', 's', 't', 'r', 'o', 'n', 'o', 'm', 'i', 'a', '.', ' ', 'T', 'e', 'x', 't', 'o', ':', ' ', '<SOS>', 'A', 's', 't', 'r', 'o', 'n', 'o', 'm', 'i', 'a', ' ', 'é', ' ', 'u', 'm', 'a', ' ', 'c', 'i', 'ê', 'n', 'c', 'i', 'a', ' ', 'n', 'a', 't', 'u', 'r', 'a', 'l', ' ', 'q', 'u', 'e', ' ', 'e', 's', 't', 'u', 'd', 'a', ' ', 'c', 'o', 'r', 'p', 'o', 's', ' ', 'c', 'e', 'l', 'e', 's', 't', 'e', 's', ' ', '(', 'c', 'o', 'm', 'o', ' ', 'e', 's', 't', 'r', 'e', 'l', 'a', 's', ',', ' ', 'p', 'l', 'a', 'n', 'e', 't', 'a', 's', ',', ' ', 'c', 'o', 'm', 'e', 't', 'a', 's', ',', ' ', 'n', 'e', 'b', 'u', 'l', 'o', 's', 'a', 's', ',', ' ', 'a', 'g', 'l', 'o', 'm', 'e', 'r', 'a', 'd', 'o', 's', ' ', 'd', 'e', ' ', 'e', 's', 't', 'r', 'e', 'l', 'a', 's', ',', ' ', 'g', 'a', 'l', 'á', 'x', 'i', 'a', 's', ')', ' ', 'e', ' ', 'f', 'e', 'n', 'ô', 'm', 'e', 'n', 'o', 's', ' ', 'q', 'u', 'e', ' ', 's', 'e', ' ', 'o', 'r', 'i', 'g', 'i', 'n', 'a', 'm', ' ', 'f', 'o', 'r', 'a', 

In [ ]:
class DatasetWikipedia(Dataset):
    def __init__(self, data_path='dataset_text/wiki_pt_0-50k.parquet', sequence_length=512, split='train'):
        self.dataset = pd.read_parquet(data_path).reset_index()  # load_dataset("wikimedia/wikipedia", "20231101.en", split=split)
        self.dataset = self.dataset.loc[self.dataset['split'] == split]
        self.sequence_length = sequence_length
        self.tokenizer = TokenizerChar()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset.text[idx]
        current_sequence = self.tokenizer.encode_text(text)
        if len(current_sequence) < self.sequence_length + 1:
            current_sequence += [self.tokenizer.pad_token_idx()] * (self.sequence_length + 1 - len(current_sequence))
        else:
            current_sequence = current_sequence[:self.sequence_length + 1]
        x = torch.tensor(current_sequence[:-1])
        y = torch.tensor(current_sequence[1:])
        return x, y

In [ ]:
dataset_train = DatasetWikipedia('dataset_text/wiki_pt_0-50k_train.parquet', sequence_length=1024, split='train')

In [ ]:
dataloader_train = DataLoader(dataset_train, batch_size=4, shuffle=True)

In [ ]:
dataloader_train.__len__()

101339

#### Fine-tuning dataset

In [ ]:
class DatasetFinancial(Dataset):
    def __init__(self, data_path='dataset_text/train.json', sequence_length=4096):  # length que engloba todo o contexto (percentil 99)
        self.data_path = data_path
        self.sequence_length = sequence_length
        self.tokenizer = TokenizerChar()
        with open(self.data_path, encoding='utf-8') as f:
            lines = f.readlines()
        self.lines = [line for line in tqdm(lines) if line.isascii()]

    def __len__(self):
        return len(self.lines)

    def readline_as_dict(self, line_number):
        try:
            return json.loads(self.lines[line_number])
        except json.JSONDecodeError:
            return dict()

    def __getitem__(self, line_idx):
        line_dict = self.readline_as_dict(line_idx)
        if len(line_dict.keys()) == 0:
            x = torch.tensor([self.tokenizer.pad_token_idx()] * self.sequence_length)
            y = torch.tensor([self.tokenizer.pad_token_idx()] * self.sequence_length)
            return x, y
        system_text = line_dict['system']
        user_text = line_dict['user']
        assistant_text = line_dict['assistant'] 
        current_sequence = (
            [self.tokenizer.sos_token_idx()]
            + [self.tokenizer.encode(c) for c in system_text]
            + [self.tokenizer.encode(' ')]
            + [self.tokenizer.encode(c) for c in user_text]
            + [self.tokenizer.eop_token_idx()]
            + [self.tokenizer.encode(c) for c in assistant_text]
            + [self.tokenizer.eos_token_idx()]
        )
        if len(current_sequence) < self.sequence_length + 1:
            current_sequence += [self.tokenizer.pad_token_idx()] * (self.sequence_length + 1 - len(current_sequence))
        else:
            current_sequence = current_sequence[:self.sequence_length + 1]
        x = torch.tensor(current_sequence[:-1])
        y = torch.tensor(current_sequence[1:])
        return x, y

In [ ]:
dataset_financial = DatasetFinancial()

100%|██████████| 518182/518182 [00:00<00:00, 3573334.53it/s]


In [ ]:
dataset_financial[0]

(tensor([257,  10,  32,  ...,   0,   0,   0]),
 tensor([10, 32, 69,  ...,  0,  0,  0]))

In [ ]:
len(dataset_financial)

442677

## Model Training

### Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
writer = SummaryWriter(log_dir='runs/transformer_experiment_1024_tokens_v2_20250824')
sequence_length = 1024
batch_size = 4
accumulation_steps = 8
dataset_train = DatasetWikipedia('dataset_text/wiki_pt_0-50k_train.parquet', sequence_length=sequence_length, split='train')
dataset_test = DatasetWikipedia('dataset_text/wiki_pt_0-50k_test.parquet', sequence_length=sequence_length, split='test')
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=True)
vocab_size = dataset_train.tokenizer.get_vocab_size()

d_model = 640
num_layers = 8
num_heads = 10
d_ff = 2048
dropout = 0.1
max_seq_length = sequence_length
# model = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)
model = TransformerDecoder(vocab_size, d_model, max_seq_length, num_layers, d_ff, num_heads, dropout=0.1)
model.to(device)
load_from_checkpoint = True

if load_from_checkpoint:
    checkpoint_path = 'model_checkpoints/model_checkpoint_4.pth'  # ajuste para o arquivo desejado
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

tgt_mask = (1 - torch.triu(
  torch.ones(1, sequence_length, sequence_length), diagonal=1)
).bool()

def init_weights(module):
    if isinstance(module, (nn.Linear)):
        init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        if module.bias is not None:
            init.zeros_(module.bias)
if not load_from_checkpoint:
    model.apply(init_weights)

optimizer = Adam(model.parameters(), lr=0.75*1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
n_epochs = 1
n_batches = int(dataset_train.__len__() // batch_size)

print("Starting model training...")
start_epoch = 4
optimizer.zero_grad()
for epoch in range(start_epoch, start_epoch + n_epochs):
    print(f"Epoch: {epoch + 1}")
    avg_loss = 0
    model.train()
    for batch_idx, batch in enumerate(tqdm(dataloader_train, total=n_batches)):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        outputs = model(x, tgt_mask.to(device))
        loss = loss_fn(outputs.view(-1, vocab_size), y.view(-1))
        avg_loss += loss.item()
        loss = loss / accumulation_steps
        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        writer.add_scalar('Loss/train', loss.item() * accumulation_steps, epoch * n_batches + batch_idx)

    torch.save(model.state_dict(), f'model_checkpoints/model_checkpoint_{epoch+1}.pth')

    avg_loss /= (batch_idx + 1)
    print(f"Average epoch training loss: {avg_loss}")
    print(f"Last batch training loss: {loss * accumulation_steps}")

    model.eval()
    avg_loss = 0
    for batch_idx, batch in enumerate(tqdm(dataloader_test, total=dataloader_test.__len__())):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        outputs = model(x, tgt_mask.to(device))
        loss = loss_fn(outputs.view(-1, vocab_size), y.view(-1))
        avg_loss += loss.item()
    
    avg_loss /= (batch_idx + 1)
    print(f"Epoch validation loss: {avg_loss}")
    writer.add_scalar('Loss/val', avg_loss, epoch)

writer.close()

cuda
Starting model training...
Epoch: 5


101339it [8:58:59,  3.13it/s]                             


Average epoch training loss: 1.1394766390121895
Last batch training loss: 1.004889726638794


100%|██████████| 1049/1049 [02:03<00:00,  8.52it/s]

Epoch validation loss: 1.0872952146571062


In [ ]:
def make_tgt_mask(sequence_length, device):
    tgt_mask = torch.tril(torch.ones(sequence_length, sequence_length, dtype=torch.bool)).to(device)
    return tgt_mask

In [ ]:
import torch

def predict_fernando(start_text, model, tokenizer, max_sequence_length=1024, temperature=1.0):
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    sequence = [tokenizer.sos_token_idx()] + [tokenizer.encode(c) for c in start_text]

    input_tokens = torch.tensor(sequence, dtype=torch.long).unsqueeze(0).to(device) # acrescenta dimensão batch_size=1
    current_text = start_text

    with torch.no_grad():
        for _ in range(len(start_text), max_sequence_length):
            outputs = model(input_tokens, tgt_mask=make_tgt_mask(input_tokens.shape[1], device))
            log_probs = outputs[0, -1] / temperature

            predicted_token_idx = torch.distributions.Categorical(logits=log_probs).sample().item()

            if predicted_token_idx == tokenizer.eos_token_idx():
                break

            current_text += tokenizer.decode(predicted_token_idx)
            input_tokens = torch.cat((input_tokens, torch.tensor([[predicted_token_idx]]).to(device)), dim=1)

    print('Texto predito:', current_text)
    return current_text


In [ ]:
predict_fernando(start_text="Tema: América. Texto: <SOS> A ", tokenizer=TokenizerChar(), model=model)

Texto predito: Tema: América. Texto: <SOS> A abriga animal é um acumulado de gênero significativo de chuva da província de Santa Cruz do Sul Su


'Tema: América. Texto: <SOS> A abriga animal é um acumulado de gênero significativo de chuva da província de Santa Cruz do Sul Su'

In [ ]:
predict_fernando(start_text="Tema: América. Texto: <SOS> A ", tokenizer=TokenizerChar(), model=model)

Texto predito: Tema: América. Texto: <SOS> A América (2 e 1) é um terminal com cerca de 1.420 metros acima do nível do mar. O território francê


'Tema: América. Texto: <SOS> A América (2 e 1) é um terminal com cerca de 1.420 metros acima do nível do mar. O território francê'

In [ ]:
predict_fernando(start_text="Alagoas", tokenizer=TokenizerChar(), model=model)

Texto predito: Alagoas (1986), são herbáceos (1980), que formam uma linguagem molecular: ela estava junto a recuperar na novela, a gradualma ao


'Alagoas (1986), são herbáceos (1980), que formam uma linguagem molecular: ela estava junto a recuperar na novela, a gradualma ao'

In [ ]:
predict_fernando(start_text="A América Latina é ", tokenizer=TokenizerChar(), model=model)

Texto predito: A América Latina é uma comuna italiana da região da Campania, província de Perna, com cerca 3.518 habitantes. Estende-se por uma área de 7 km², tendo uma densidade populacional de 44 hab/km². Faz fronteiro com Cagmasata, Campania, Castalno, Corliano, San Riosini, Rosaazle, São Rongellindo.

Demografia

Comunas de Perna (província)


'A América Latina é uma comuna italiana da região da Campania, província de Perna, com cerca 3.518 habitantes. Estende-se por uma área de 7 km², tendo uma densidade populacional de 44 hab/km². Faz fronteiro com Cagmasata, Campania, Castalno, Corliano, San Riosini, Rosaazle, São Rongellindo.\n\nDemografia\n\nComunas de Perna (província)'

In [ ]:
predict_fernando(start_text="Portugal é ", tokenizer=TokenizerChar(), model=model)

Texto predito: Portugal é um município brasileiro do estado de Galiza, situado na Região Paulista, Região Metropolitana do Rio Grande do Sul em Belo, Região Metropolitana, Região Metropolitana do Centro Estadual de Santa Cruz, Rio Grande do Sul. Recentemente em 1895, enquanto extinguiu ao Rio Grande do Sul, período do Maranhão. Conta-se em 1898, iniciou-se com o município pertencente à Região Metropolitana do Rio Grande do Sul, sendo a 2.ª cesquisa do município de Galiza, ao norte da Região Metropolitana do Sul e a mois censo de dois municípios.

Ver também 
 Lista de municípios de Galiza
 Caldas Amorais
 Hospitalismo da Rio Grande do Sul
 Luga de Suzana
 Lista de estilos do século I a.C.


'Portugal é um município brasileiro do estado de Galiza, situado na Região Paulista, Região Metropolitana do Rio Grande do Sul em Belo, Região Metropolitana, Região Metropolitana do Centro Estadual de Santa Cruz, Rio Grande do Sul. Recentemente em 1895, enquanto extinguiu ao Rio Grande do Sul, período do Maranhão. Conta-se em 1898, iniciou-se com o município pertencente à Região Metropolitana do Rio Grande do Sul, sendo a 2.ª cesquisa do município de Galiza, ao norte da Região Metropolitana do Sul e a mois censo de dois municípios.\n\nVer também \n Lista de municípios de Galiza\n Caldas Amorais\n Hospitalismo da Rio Grande do Sul\n Luga de Suzana\n Lista de estilos do século I a.C.'